# Decomposition and seasonality: lab notebook

**Statistical Modelling and Forecasting, Statistics background for forecasting**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose, STL
import matplotlib.pyplot as plt

### 1. The numerical problem: classical additive decomposition by hand (m = 4)

In [ ]:
# 1. The numerical problem: classical additive decomposition by hand (m = 4)
y = np.array([47, 58, 70, 49, 51, 64, 75, 55, 56, 66, 80, 59], dtype=float)
T = np.full(12, np.nan)
for t in range(2, 10):
    T[t] = (0.5 * y[t - 2] + y[t - 1] + y[t] + y[t + 1] + 0.5 * y[t + 2]) / 4
D = y - T
raw = np.array([np.nanmean(D[q::4]) for q in range(4)])
adj = raw - raw.mean()
print("trend  :", np.round(T, 2))
print("raw seasonal indices :", np.round(raw, 3))
print("adjusted indices     :", np.round(adj, 3), "| sum =", round(adj.sum(), 6))
print("seasonally adjusted y1:", round(y[0] - adj[0], 3))

### 2. The same with statsmodels

In [ ]:
# 2. The same with statsmodels
s = pd.Series(y, index=pd.period_range("2021Q1", periods=12, freq="Q").to_timestamp())
res = seasonal_decompose(s, model="additive", period=4)
print("statsmodels indices:", np.round(res.seasonal.values[:4], 3))

### 3. Real data: monthly CO2, additive and multiplicative

In [ ]:
# 3. Real data: monthly CO2, additive and multiplicative
co2 = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
add = seasonal_decompose(co2, model="additive", period=12)
mul = seasonal_decompose(co2, model="multiplicative", period=12)
print("additive seasonal range    :", round(add.seasonal.max() - add.seasonal.min(), 2), "ppm")
print("multiplicative index range :", round(mul.seasonal.min(), 4), "to", round(mul.seasonal.max(), 4))
print("residual std additive:", round(add.resid.std(), 3), "| multiplicative (as ppm):", round((mul.resid.dropna() * mul.trend.dropna()).std(), 3))
print("month with the highest value:", int(add.seasonal.iloc[:12].values.argmax()) + 1)

### 4. Seasonal differencing removes the seasonal pattern without estimating indices

In [ ]:
# 4. Seasonal differencing removes the seasonal pattern without estimating indices
sd12 = co2.diff(12).dropna()
print("std raw:", round(co2.std(), 2), "| after lag-12 difference:", round(sd12.std(), 3))

### 5. STL lets the seasonal pattern drift slowly

In [ ]:
# 5. STL lets the seasonal pattern drift slowly
stl = STL(co2, period=12, robust=True).fit()
print("STL seasonal std:", round(stl.seasonal.std(), 3), "| STL remainder std:", round(stl.resid.std(), 3))
add.plot(); plt.show()

## Expected output

Running all cells above should print:

```
trend  : [  nan   nan 56.5  57.75 59.12 60.5  61.88 62.75 63.62 64.75   nan   nan]
raw seasonal indices : [-7.875  2.375 13.312 -8.25 ]
adjusted indices     : [-7.766  2.484 13.422 -8.141] | sum = 0.0
seasonally adjusted y1: 54.766
statsmodels indices: [-7.766  2.484 13.422 -8.141]
additive seasonal range    : 6.07 ppm
multiplicative index range : 0.9907 to 1.0086
residual std additive: 0.283 | multiplicative (as ppm): 16.667
month with the highest value: 3
std raw: 17.12 | after lag-12 difference: 0.674
STL seasonal std: 2.026 | STL remainder std: 0.255
```

## Practice exercises

Try these in the cells below.

1. Switch between additive and multiplicative with a steep trend. Which mode produces a remainder without leftover pattern?
2. Increase the noise. How does the quality of the estimated seasonal indices (compare with the true ones) change?
3. Change the season length from 12 to 4 and read off the four indices.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
